# GĐ 2: Retrain Pipeline on SemEval 2016 Only

**Dataset:** SemEval 2016 SB1 Restaurant (train 2,000 sent / 2,507 ops, test 676 sent / 859 ops)

**Experiments:**
1. **Retrieval + sqrt weights** — full pipeline (embedding + FAISS + ABSA)
2. **No-retrieval ablation** — baseline without retrieval

**Config:** sqrt class weights `[1.00, 1.70, 4.33]`, DeBERTa-v3-base

**Kaggle:** T4 GPU, ~3 hours total

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')

print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
KAGGLE_INPUT = '/kaggle/input/semeval-absa-restaurant'

os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)

shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')

print('SemEval 2016 SB1 XML files in place (train + test only).')

## 1. Data Preparation

In [ ]:
!python scripts/01_prepare_data.py

## 2. Train Contrastive Embedding

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
import yaml

emb_cfg = {
    'model_name': 'microsoft/deberta-v3-base',
    'proj_dim': 256,
    'tau': 0.07,
    'batch_size': 32,
    'epochs': 15,
    'lr': 2.0e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'max_seq_length': 128,
    'grad_clip': 1.0,
    'patience': 5,
    'seed': 42,
    'triplets_path': 'data/processed/contrastive_triplets.jsonl',
    'val_ratio': 0.1,
    'ckpt_dir': 'checkpoints/embedding',
    'log_path': 'logs/embedding_training.jsonl',
}

with open('configs/embedding_v2.yaml', 'w') as f:
    yaml.dump(emb_cfg, f)

print('Embedding config: batch=32, epochs=15, patience=5')

In [ ]:
!python scripts/02_train_embedding.py --config configs/embedding_v2.yaml

In [ ]:
if os.path.exists('logs/embedding_training.jsonl'):
    with open('logs/embedding_training.jsonl') as f:
        for line in f:
            rec = json.loads(line)
            print(f"Epoch {rec.get('epoch', '?')}: loss={rec.get('train_loss', 0):.4f}, "
                  f"R@1={rec.get('recall@1', 0):.4f}, R@3={rec.get('recall@3', 0):.4f}, "
                  f"R@5={rec.get('recall@5', 0):.4f}")

## 3. Build FAISS Index

In [ ]:
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --input data/processed/classification.jsonl \
    --bio_input data/processed/bio_tagging.jsonl \
    --out_dir indexes/

## 4. Train ABSA

### 4A. Retrieval + sqrt class weights (main experiment)

In [ ]:
!python scripts/04_train_absa.py \
    --config configs/absa_exp_c.yaml \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --grad_accum_steps 8 \
    --ckpt_path checkpoints/absa/exp_retrieval.pt

### 4B. No-retrieval ablation (baseline)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

!python scripts/04_train_absa.py \
    --config configs/absa_exp_c.yaml \
    --no_retrieval \
    --grad_accum_steps 8 \
    --ckpt_path checkpoints/absa/exp_no_retrieval.pt

## 5. Evaluate

In [ ]:
print('='*60)
print('EXPERIMENT: Retrieval + sqrt class weights')
print('='*60)
!python scripts/05_evaluate.py \
    --config configs/absa_exp_c.yaml \
    --checkpoint checkpoints/absa/exp_retrieval.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/

In [ ]:
print('='*60)
print('ABLATION: No-retrieval baseline')
print('='*60)
!python scripts/05_evaluate.py \
    --config configs/absa_exp_c.yaml \
    --checkpoint checkpoints/absa/exp_no_retrieval.pt \
    --no_retrieval

## 6. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_gd2'
os.makedirs(output_dir, exist_ok=True)

for src, dst_name in [
    ('checkpoints/embedding/best.pt', 'embedding_best.pt'),
    ('checkpoints/absa/exp_retrieval.pt', 'absa_retrieval.pt'),
    ('checkpoints/absa/exp_no_retrieval.pt', 'absa_no_retrieval.pt'),
    ('indexes/train.faiss', 'train.faiss'),
    ('indexes/train_vectors.npy', 'train_vectors.npy'),
    ('indexes/train_metadata.jsonl', 'train_metadata.jsonl'),
]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(output_dir, dst_name))
        size_mb = os.path.getsize(src) / 1e6
        print(f'{dst_name}: {size_mb:.1f} MB')

if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(output_dir, 'logs'), dirs_exist_ok=True)

print(f'\nAll outputs saved to {output_dir}')